In [6]:
import numpy as np
import random  
import tensorflow as tf
from datetime import datetime

In [7]:
class PPOAgent:
    def __init__(self, state_dim, action_dim, learning_rate=0.001, gamma=0.95, clip_ratio=0.2):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.clip_ratio = clip_ratio
        self.policy = self.build_policy_network()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(self.state_dim,)),
            tf.keras.layers.Dense(self.action_dim, activation='softmax')
        ])
        return model

    def get_action_prob(self, state):
        return self.policy.predict(np.array([state]))[0]

    def train(self, states, actions, advantages):
        actions = tf.keras.utils.to_categorical(actions, self.action_dim)
        advantages = np.array(advantages)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - self.clip_ratio, 1 + self.clip_ratio)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))


In [8]:
class HomeAutomation:
    def __init__(self):
        self.lights = []
        self.tvs = []
        self.fans = []
        self.acs = []
        self.ppo_agent = PPOAgent(state_dim=6, action_dim=4)  # Initialize PPO agent

    def add_light(self, RGB: bool, Ambient: bool, location: str):
        light = Light(RGB, Ambient, location)
        self.lights.append(light)

    def add_tv(self, location: str, power_state: bool, volume: int, channel: int):
        tv = TV(location, power_state, volume, channel)
        self.tvs.append(tv)

    def add_fan(self, fan_speed: int, location: str):
        fan = Fan(fan_speed, location)
        self.fans.append(fan)

    def add_ac(self, location: str, temperature: int):
        ac = AC(location, temperature)
        self.acs.append(ac)

    def fetch_status(self):
        status = {
            "lights": [light.fetch_status() for light in self.lights],
            "tvs": [tv.fetch_status() for tv in self.tvs],
            "fans": [fan.fetch_status() for fan in self.fans],
            "acs": [ac.fetch_status() for ac in self.acs]
        }
        return status

    def control_lights(self, location: str, value: int):
        for light in self.lights:
            if light.location == location:
                light.control_normal_light(value)
                break
        else:
            print(f"No light found at location {location}")

    def control_tv(self, location: str, power_state: bool, volume: int, channel: int):
        for tv in self.tvs:
            if tv.location == location:
                tv.control_tv(power_state, volume, channel)
                break
        else:
            print(f"No TV found at location {location}")

    def control_fan(self, location: str, speed: int):
        for fan in self.fans:
            if fan.location == location:
                fan.control_fan(speed)
                break
        else:
            print(f"No fan found at location {location}")

    def control_ac(self, location: str, temperature: int):
        for ac in self.acs:
            if ac.location == location:
                ac.control_ac(temperature)
                break
        else:
            print(f"No AC found at location {location}")

class Light:
    def __init__(self, RGB: bool, Ambient: bool, location: str, *args, **kwargs):
        self.RGB = RGB
        self.Ambient = Ambient
        self.location = location
        self.light = None  # Initialize light state as None (off)

    def control_normal_light(self, value: int):
        if not self.RGB and not self.Ambient:
            if value in [0, 1]:
                self.light = value
                print(f"Normal light in {self.location} turned {'off' if not value else 'on'}")
            else:
                print(f"Invalid value for normal light control: {value}")
        else:
            print(f"This method is not applicable for {self.get_type()} light.")

    def control_ambient_light(self, brightness: int):
        if self.Ambient:
            if 0 <= brightness <= 100:
                self.light = brightness
                print(f"Ambient light in {self.location} set to brightness {brightness}")
            else:
                print(f"Invalid brightness value for ambient light: {brightness}")
        else:
            print(f"This method is not applicable for {self.get_type()} light.")

    def control_rgb_light(self, r: int, g: int, b: int):
        if self.RGB:
            if 0 <= r <= 255 and 0 <= g <= 255 and 0 <= b <= 255:
                self.light = {"r": r, "g": g, "b": b}
                print(f"RGB light in {self.location} set to color ({r}, {g}, {b})")
            else:
                print(f"Invalid color values for RGB light: (r: {r}, g: {g}, b: {b})")
        else:
            print(f"This method is not applicable for {self.get_type()} light.")

    def fetch_status(self) -> dict:
        status = {"location": self.location}
        if self.RGB:
            status["type"] = "RGB"
            status["color"] = self.light
        elif self.Ambient:
            status["type"] = "Ambient"
            status["brightness"] = self.light
        else:
            status["type"] = "Normal"
            status["state"] = "On" if self.light else "Off"
        return status

    def get_type(self) -> str:
        if self.RGB:
            return "RGB"
        elif self.Ambient:
            return "Ambient"
        else:
            return "Normal"

class TV:
    def __init__(self, location: str, power_state: bool, volume: int, channel: int):
        self.location = location
        self.power_state = power_state
        self.volume = volume
        self.channel = channel

    def control_tv(self, power_state: bool, volume: int, channel: int):
        self.power_state = power_state
        self.volume = volume
        self.channel = channel

    def fetch_status(self) -> dict:
        return {"Current Status of TV": {"Location": self.location, "Power State": self.power_state, "Volume": self.volume, "Channel": self.channel}}

class Fan:
    def __init__(self, fan_speed: int, location: str, *args, **kwargs):
        self.fan_speed = fan_speed
        self.location = location

    def control_fan(self, speed: int):
        self.fan_speed = speed

    def fetch_status(self) -> dict:
        return {"Fan_Speed": self.fan_speed, "Location": self.location}

class AC:
    def __init__(self, location: str, temperature: int):
        self.location = location
        self.temperature = temperature

    def control_ac(self, temperature: int):
        if 18 <= temperature <= 30:
            self.temperature = temperature
            print(f"AC at {self.location} set to {temperature}°C")
        else:
            print(f"Invalid temperature range. Please set between 18°C and 30°C.")

    def fetch_status(self) -> dict:
        status = {
            "location": self.location,
            "temperature": self.temperature,
            "power": "ON" if self.temperature != 0 else "OFF",
        }
        return status



In [9]:
# Create an instance of HomeAutomation
home_auto = HomeAutomation()

# Add lights
home_auto.add_light(RGB=False, Ambient=False, location="Living Room")
home_auto.add_light(RGB=True, Ambient=False, location="Bedroom")

# Add TVs
home_auto.add_tv(location="Living Room", power_state=False, volume=50, channel=1)
home_auto.add_tv(location="Bedroom", power_state=True, volume=30, channel=2)

# Add fans
home_auto.add_fan(fan_speed=3, location="Living Room")
home_auto.add_fan(fan_speed=2, location="Bedroom")

# Add ACs
home_auto.add_ac(location="Living Room", temperature=25)
home_auto.add_ac(location="Bedroom", temperature=20)

# Fetch status
status = home_auto.fetch_status()
print("Current Status:")
print(status)

# Control appliances
home_auto.control_lights(location="Living Room", value=1)
home_auto.control_tv(location="Bedroom", power_state=True, volume=40, channel=3)
home_auto.control_fan(location="Living Room", speed=1)
home_auto.control_ac(location="Bedroom", temperature=22)

# Fetch status again
updated_status = home_auto.fetch_status()
print("\nUpdated Status:")
print(updated_status)

AttributeError: module 'tensorflow' has no attribute 'keras'

In [ ]:
!pip install tensorflow keras